<a href="https://colab.research.google.com/github/Pentaphone/nn_tests/blob/main/nn_test2/hyperparam_optim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperparameter Optimization

### Setup

In [ ]:
# INSTALLS, IMPORTS
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

!pip install torch_geometric
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

!pip install optuna
import optuna

!pip install kagglehub
import kagglehub

!pip install rdkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from rdkit.Chem import rdForceFieldHelpers
from rdkit.Chem import AllChem

!pip install py3Dmol
import py3Dmol


# DOWNLOAD DATA
data_path = kagglehub.dataset_download("markinsuff/lipophilicity") + "/Lipophilicity.csv"
print("Path to dataset:", data_path)


# CONFIG
n_trials = 100
timeout = 60 * 60 * 6  # seconds

limit = 100
patience = 7

final_training_limit = 200
final_training_patience = 13

save_path = "/content"

SEED = 42
torch.manual_seed(SEED)

Using Colab cache for faster access to the 'lipophilicity' dataset.
Path to dataset: /kaggle/input/lipophilicity/Lipophilicity.csv


### Utility Functions

In [ ]:
# UTIL

def graph_from_mol(mol):
  node_features = []
  edge_indices = []

  for atom in mol.GetAtoms():
    features = [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetTotalNumHs(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
    ]
    node_features.append(features)

  node_features = torch.tensor(node_features, dtype=torch.float)

  for bond in mol.GetBonds():
    start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    edge_indices.append([start, end])
    edge_indices.append([end, start])

  edge_indices = torch.tensor(edge_indices, dtype=torch.long)
  # transpose for torch_geometric
  edge_indices = edge_indices.t().contiguous()

  return Data(x=node_features, edge_index=edge_indices)


def mol_3d_from_mol(mol):
  mol = Chem.AddHs(mol)
  AllChem.EmbedMolecule(mol)
  # optimize molecule geometry with Merck molecular force field (MMFF)
  AllChem.MMFFOptimizeMolecule(mol)

  mol_3d = Chem.MolToMolBlock(mol)
  return mol_3d


### Data

In [ ]:
# DATA
class MoleculeDataset(Dataset):
  def __init__(self, csv_file):
    self.data = pd.read_csv(csv_file)
    self.normalize = False
    self.mean, self.std = self.data["exp"].mean(), self.data["exp"].std()

  def __len__(self):
    return len(self.data)

  def normalize_target(self, value: bool):
    self.normalize = value

  def __getitem__(self, i):
    smiles = self.data.loc[i, "RDKIT_SMILES"]
    mol = Chem.MolFromSmiles(smiles)
    graph = graph_from_mol(mol)

    exp = self.data.loc[i, "exp"]
    if self.normalize:
      exp = (exp - self.mean) / self.std

    return graph, exp


dataset = MoleculeDataset(data_path)
dataset.normalize_target(True)

data_std = dataset.std

# split data
train_data, test_data = torch.utils.data.random_split(dataset, [0.85, 0.15],
  generator=torch.Generator().manual_seed(SEED))


### Model

In [ ]:
# MODEL
class GraphNeuralNetwork(nn.Module):
  def __init__(self, dropout_rate, n_conv_layers, hidden_dim):
    super(GraphNeuralNetwork, self).__init__()

    self.conv_layers = nn.ModuleList()
    self.conv_layers.append(GCNConv(5, hidden_dim))

    self.batch_norm_layers = nn.ModuleList()
    self.batch_norm_layers.append(nn.BatchNorm1d(hidden_dim))

    for _ in range(n_conv_layers - 1):
      self.conv_layers.append(GCNConv(hidden_dim, hidden_dim))
      self.batch_norm_layers.append(nn.BatchNorm1d(hidden_dim))

    self.pool = global_mean_pool

    self.linear = nn.Sequential(
      nn.Linear(hidden_dim, hidden_dim // 2),
      nn.ReLU(),
      nn.Dropout(dropout_rate),
      nn.Linear(hidden_dim // 2, 1),
    )

  def forward(self, x, edge_index, batch):
    for i in range(len(self.conv_layers)):
      x = self.conv_layers[i](x, edge_index)
      x = self.batch_norm_layers[i](x)
      x = torch.relu(x)

    x = self.pool(x, batch)
    x = self.linear(x)
    return x

  def predict(self, mol):
    with torch.no_grad():
      graph = graph_from_mol(mol)
      if getattr(graph, 'batch', None) is None:
        graph.batch = torch.zeros(graph.x.shape[0], dtype=torch.long)
      output = self(graph.x, graph.edge_index, graph.batch)
      logd = output.item() * data_std + dataset.mean
      return logd


### Training and Testing

In [ ]:
# TRAINING AND TESTING

def train(model, data_loader, loss_fn, optimizer, verbose=True):
  model.train()
  for i, (batch_data, target) in enumerate(data_loader):
    optimizer.zero_grad()
    output = model(batch_data.x, batch_data.edge_index, batch_data.batch)
    loss = loss_fn(output, target.unsqueeze(1))
    loss.backward()
    optimizer.step()

    if verbose and i % 10 == 0:
      loss_val = loss.item()
      current_samples = min((i + 1) * data_loader.batch_size, len(data_loader.dataset))
      print(f"loss: {loss_val:>7f}  [{current_samples:>5d}/{len(data_loader.dataset):>5d}]")

  print()

def evaluate(model, data_loader, loss_fn, dataset_name="", verbose=True):
  model.eval()
  total_loss = 0
  with torch.no_grad():
    for i, (batch_data, target) in enumerate(data_loader):
      output = model(batch_data.x, batch_data.edge_index, batch_data.batch)
      total_loss += loss_fn(output, target.unsqueeze(1)).item()

  avg_loss = total_loss / len(data_loader)

  if verbose:
    print(f"Average loss ({dataset_name}): {avg_loss:>8f}")
    rmse_logd = avg_loss**0.5 * data_std
    print(f"RMSE LogD ({dataset_name}): {rmse_logd:>8f}\n")

  return avg_loss

### Hyperparameter Optimization

In [ ]:
# HYPERPARAMETER OPTIMIZATION

def objective(trial):
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
  n_conv_layers = trial.suggest_int("n_conv_layers", 2, 4)
  hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])

  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

  model = GraphNeuralNetwork(dropout_rate, n_conv_layers, hidden_dim)

  loss_fn = nn.MSELoss()
  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

  best_test_loss = float("inf")
  epochs_without_improvement = 0

  for epoch in range(limit):
    model.train()
    for batch_data, target in train_loader:
      optimizer.zero_grad()
      output = model(batch_data.x, batch_data.edge_index, batch_data.batch)
      loss = loss_fn(output, target.unsqueeze(1))
      loss.backward()
      optimizer.step()

    current_test_loss = evaluate(model, test_loader, loss_fn, verbose=False)
    lr_scheduler.step(current_test_loss)

    if current_test_loss < best_test_loss:
      best_test_loss = current_test_loss
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
      break

    trial.report(best_test_loss, epoch)

    # pruning
    if trial.should_prune():
      raise optuna.exceptions.TrialPruned()

  return best_test_loss


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)

print("Best trial:")
trial = study.best_trial
print("  Loss ():", trial.value)
print("  RMSE logD:", trial.value**0.5 * data_std)

print("Best parameters:")
for key, value in trial.params.items():
  print(f"  {key}: {value}")

[I 2026-08-18 02:38:31,590] A new study created in memory with name: no-name-34109b9f-fc90-47e4-9d5a-7428049928bd


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-18 02:46:11,975] Trial 0 finished with value: 0.7625217406079173 and parameters: {'learning_rate': 5.6115164153345e-05, 'weight_decay': 0.0007969454818643932, 'dropout_rate': 0.39279757672456206, 'batch_size': 16, 'n_conv_layers': 2, 'hidden_dim': 32}. Best is trial 0 with value: 0.7625217406079173.
[I 2026-08-18 02:53:54,423] Trial 1 finished with value: 1.001611102372408 and parameters: {'learning_rate': 1.0994335574766187e-05, 'weight_decay': 0.0008706020878304854, 'dropout_rate': 0.4329770563201687, 'batch_size': 16, 'n_conv_layers': 2, 'hidden_dim': 32}. Best is trial 0 with value: 0.7625217406079173.
[I 2026-08-18 02:57:32,713] Trial 2 finished with value: 0.652037626504898 and parameters: {'learning_rate': 0.00016738085788752134, 'weight_decay': 1.9010245319870364e-05, 'dropout_rate': 0.21685785941408728, 'batch_size': 64, 'n_conv_layers': 2, 'hidden_dim': 64}. Best is trial 2 with value: 0.652037626504898.
[I 2026-08-18 03:02:04,132] Trial 3 finished with value: 0.52

### Final Training

In [1]:
# FINAL TRAINING

model = GraphNeuralNetwork(
    dropout_rate=trial.params["dropout_rate"],
    n_conv_layers=trial.params["n_conv_layers"],
    hidden_dim=trial.params["hidden_dim"],
)

train_loader = DataLoader(train_data, batch_size=trial.params["batch_size"], shuffle=True)
test_loader = DataLoader(test_data, batch_size=trial.params["batch_size"], shuffle=False)

loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=trial.params["learning_rate"],
    weight_decay=trial.params["weight_decay"]
)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

def train_model(
    model,
    train_loader,
    test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    patience=final_training_patience,
    limit=final_training_limit,
    plot=True,
    save=True,
):

  train_losses = []
  test_losses = []

  best_test_loss = float("inf")
  best_epoch = 0
  epochs_without_improvement = 0

  print("Initial model performane")
  train_loss = evaluate(model, train_loader, loss_fn, dataset_name="train data")
  train_losses.append(train_loss)
  test_loss = evaluate(model, test_loader, loss_fn, dataset_name="test data")
  test_losses.append(test_loss)

  for epoch in range(limit):
    print(f"Epoch {epoch + 1}/{limit}")
    train(model, train_loader, loss_fn, optimizer)

    print("Evaluating")
    train_loss = evaluate(model, train_loader, loss_fn, dataset_name="train data")
    train_losses.append(train_loss)
    test_loss = evaluate(model, test_loader, loss_fn, dataset_name="test data")
    test_losses.append(test_loss)

    lr_scheduler.step(test_loss)

    if test_loss < best_test_loss:
      torch.save(model.state_dict(), "best_model_weights.pth")
      best_test_loss = test_loss
      best_epoch = epoch
      epochs_without_improvement = 0
    else:
      epochs_without_improvement += 1

    if epochs_without_improvement >= 1:
      print(f"Epochs without improvement: {epochs_without_improvement}")

      if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch + 1}")

        print(f"Loading best model (Epoch {best_epoch + 1})\n")
        model.load_state_dict(torch.load("best_model_weights.pth"))

        break

      print()

  print("Training and testing completed")

  # total parameters
  total_params = sum(param.numel() for param in model.parameters())
  print(f"Total parameters: {total_params:,}")

  # RMSE logD
  best_rmse_logd = best_test_loss**0.5 * data_std
  print(f"RMSE predicted logD (test data): {best_rmse_logd:>8f}")
  print()

  # plot
  if plot:
    train_rmse = [loss**0.5 * data_std for loss in train_losses]
    test_rmse = [loss**0.5 * data_std for loss in test_losses]

    plt.style.use('dark_background')
    plt.figure(figsize=(4, 4))
    plt.plot(train_rmse, label="Train data")
    plt.plot(test_rmse, label="Test data")
    plt.xlabel("Epoch")
    plt.ylabel("RMSE logD")
    plt.legend()

    plt.show()
    print()

  # save
  if save:
    torch.save(model, save_path+"/model.pth")
    torch.save(model.state_dict(), save_path+"/model_weights.pth")
    print("Model saved")


train_model(model, train_loader, test_loader)


NameError: name 'GraphNeuralNetwork' is not defined

### Example Prediction

In [ ]:
# EXAMPLE
visualization = "3d"

i = torch.randint(0, len(test_data), (1,)).item()
mol = Chem.MolFromSmiles(test_data.dataset.data.loc[i, "RDKIT_SMILES"])
formula = rdMolDescriptors.CalcMolFormula(mol)
exp = test_data.dataset.data.loc[i, "exp"]

model.eval()
pred = model.predict(mol)
error = abs(pred - exp)

if visualization not in ["2d", "3d", "none"]:
  raise ValueError("Visualization must be either '2d', '3d' or 'none'")

match visualization:

  case "2d":
    plt.figure(figsize=(4,4))
    plt.axis("off")
    image = Draw.MolToImage(mol)
    plt.imshow(image)
    plt.show()

  case "3d":
    view = py3Dmol.view(width=400, height=400)
    view.setBackgroundColor('black')
    view.addModel(mol_3d_from_mol(mol), "mol")
    view.setStyle({
        "stick": {},
        "sphere": {"scale": 0.3}
        })
    view.zoomTo()
    view.show()

  case "none":
    pass

print(formula)
print(f"Predicted logD: {pred:.2f}")
print(f"True logD: {exp:.2f}")
print(f"Error: {error:.2f}")